In [2]:
from google.colab import files
uploaded = files.upload()

Saving bezdekIris.data to bezdekIris.data
Saving Index to Index
Saving iris.data to iris.data
Saving iris.names to iris.names


In [3]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import minimum_spanning_tree
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [4]:
filename = 'iris.data'

# Load file
raw_df = pd.read_csv(filename, header=None)

# If last column contains species names, remove it
try:
    X = raw_df.astype(float)
except:
    X = raw_df.iloc[:, :-1].astype(float)

X = X.values

print("Dataset shape:", X.shape)
print(X[:5])

Dataset shape: (150, 4)
[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]


In [5]:
def mst_clustering(X, k=3, distance_metric='euclidean'):
    # Build pairwise distance matrix
    dist_matrix = squareform(pdist(X, metric=distance_metric))

    # Construct Minimum Spanning Tree
    mst = minimum_spanning_tree(dist_matrix)
    mst = mst.toarray()

    # Extract MST edges
    edges = []
    n = mst.shape[0]

    for i in range(n):
        for j in range(n):
            if mst[i, j] > 0:
                edges.append((i, j, mst[i, j]))

    # Sort edges by descending weight
    edges_sorted = sorted(edges, key=lambda x: x[2], reverse=True)

    # Remove (k-1) largest edges
    mst_cut = mst.copy()

    for i in range(k - 1):
        u, v, w = edges_sorted[i]
        mst_cut[u, v] = 0

    # Convert to undirected graph
    mst_undirected = mst_cut + mst_cut.T

    # Find connected components
    visited = [False] * n
    labels = np.full(n, -1)
    cluster_id = 0

    def dfs(start):
        stack = [start]
        while stack:
            node = stack.pop()
            if not visited[node]:
                visited[node] = True
                labels[node] = cluster_id

                neighbors = np.where(mst_undirected[node] > 0)[0]
                for nb in neighbors:
                    if not visited[nb]:
                        stack.append(nb)

    for node in range(n):
        if not visited[node]:
            dfs(node)
            cluster_id += 1

    return labels

In [6]:
k = 3
mst_labels = mst_clustering(X, k=k, distance_metric='euclidean')

print("MST Cluster Labels:")
print(mst_labels)

MST Cluster Labels:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 2 1 1 1 1 1 1 1 1 1 1 1 1 1 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1]


In [7]:
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X)

print("K-Means Cluster Labels:")
print(kmeans_labels)

K-Means Cluster Labels:
[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2 0 2 2 2 2 0 2 2 2 2
 2 2 0 0 2 2 2 2 0 2 0 2 0 2 2 0 0 2 2 2 2 2 0 2 2 2 2 0 2 2 2 0 2 2 2 0 2
 2 0]


In [8]:
mst_df = pd.DataFrame({
    'SampleId': np.arange(len(mst_labels)),
    'ClusterLabel': mst_labels
})

kmeans_df = pd.DataFrame({
    'SampleId': np.arange(len(kmeans_labels)),
    'ClusterLabel': kmeans_labels
})

mst_df.to_csv('mst_clusters.csv', index=False)
kmeans_df.to_csv('kmeans_clusters.csv', index=False)

print("Saved mst_clusters.csv")
print("Saved kmeans_clusters.csv")

Saved mst_clusters.csv
Saved kmeans_clusters.csv


In [9]:
mst_score = silhouette_score(X, mst_labels)
kmeans_score = silhouette_score(X, kmeans_labels)

print("MST Silhouette Score:", round(mst_score, 4))
print("K-Means Silhouette Score:", round(kmeans_score, 4))

MST Silhouette Score: 0.5118
K-Means Silhouette Score: 0.5526


In [10]:
print("\n--- Comparison Report ---")

if mst_score > kmeans_score:
    print("MST clustering performs better on this dataset.")
else:
    print("K-Means performs better on this dataset.")

print("\nMST Clustering:")
print("- Works well for irregular or non-spherical cluster shapes")
print("- Useful when clusters are connected by graph structure")
print("- Can handle elongated clusters better")

print("\nK-Means:")
print("- Works better for compact, spherical clusters")
print("- Usually performs better on the Iris dataset")
print("- Faster and simpler for large datasets")


--- Comparison Report ---
K-Means performs better on this dataset.

MST Clustering:
- Works well for irregular or non-spherical cluster shapes
- Useful when clusters are connected by graph structure
- Can handle elongated clusters better

K-Means:
- Works better for compact, spherical clusters
- Usually performs better on the Iris dataset
- Faster and simpler for large datasets


In [11]:
files.download('mst_clusters.csv')
files.download('kmeans_clusters.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>